# DSPy Crash Course — Part 1: Introduction & Core Concepts

> **Reference:** https://dspy.ai  
> **What is DSPy?**  
> DSPy is a framework for *programming* — not just prompting — language models. Instead of hand-crafting brittle prompts, you write modular Python programs using **Signatures**, **Modules**, and **Optimizers**. DSPy can then automatically compile those programs into high-quality prompts or fine-tunes.

---

## Table of Contents
1. [Installation](#1-installation)  
2. [LM Setup — Connecting to Groq](#2-lm-setup)  
3. [Direct LM Calls](#3-direct-calls)  
4. [Caching & `rollout_id`](#4-caching--rollout_id)  
5. [Signatures — Inline](#5-signatures--inline)  
6. [Signatures — Class-based (Advanced)](#6-signatures--class-based)  
7. [History Inspection](#7-history-inspection)  
8. [Using the Responses API](#8-responses-api)  
9. [Quick Reference Cheatsheet](#9-cheatsheet)

---
## 1. Installation

In [1]:
# Install DSPy (and its LiteLLM backend for multi-provider support)
!pip install -q dspy-ai

In [2]:
import dspy
from dspy import LM, Signature, Predict, settings

print(f"DSPy version: {dspy.__version__}")

DSPy version: 3.1.3


---
## 2. LM Setup

DSPy uses **LiteLLM** under the hood, which means you can connect to **any provider** (OpenAI, Anthropic, Groq, Gemini, local Ollama, etc.) with the same API.

### Provider Format
```
provider/model_name
```
Examples:
| Provider | Model string |
|---|---|
| OpenAI | `openai/gpt-4o-mini` |
| Anthropic | `anthropic/claude-3-5-sonnet-20241022` |
| Groq | `groq/llama-3.3-70b-versatile` |
| Gemini | `google/gemini-2.0-flash` |
| Ollama (local) | `ollama/llama3.2` |

> **Tip:** Groq-hosted OpenAI models use a double prefix: `groq/openai/gpt-oss-20b`  
> This lets DSPy distinguish Groq-hosted from native OpenAI models.

In [3]:
from google.colab import userdata

# --- Groq Setup ---
groq_api_key = userdata.get('GROQ_API')
api_base     = 'https://api.groq.com/openai/v1'

# Model name: "groq/openai/..." tells LiteLLM to use Groq + OpenAI-compatible model
model = "groq/openai/gpt-oss-20b"

print("API key loaded:", "✓" if groq_api_key else "✗ (check Colab secrets)")

API key loaded: ✓


In [4]:
# Initialize the LM object
lm = LM(
    model=model,
    api_base=api_base,
    api_key=groq_api_key,
)

# Set it as the global default — all DSPy modules will use this LM unless overridden
settings.configure(lm=lm)

print("LM configured:", lm.model)

LM configured: groq/openai/gpt-oss-20b


---
## 3. Direct LM Calls

You can call the `lm` object directly — it returns a list of output strings.  
This is useful for quick tests and understanding the low-level API before using higher-level modules.

In [5]:
# --- Method 1: Plain string prompt ---
output = lm("Hi there, I am using DSPy!")
print(type(output))   # list of strings
print(output)

# --- Method 2: OpenAI-style messages ---
output2 = lm(messages=[{"role": "user", "content": "What is 2 + 2?"}])
print(output2)

<class 'list'>
['Hey there! 👋  \n\nIt’s great to hear you’re diving into **DSPy**. That library is a fantastic way to build composable, modular language‑model workflows in pure Python. How can I help you get the most out of it today?\n\n- **Getting Started** – Need a quick “Hello World” style DSPy example?\n- **Troubleshooting** – Running into errors or unexpected outputs?\n- **Advanced Patterns** – Want to combine multiple prompts, chain LMs, or use the DSL syntax?\n- **Project Ideas** – Looking for inspiration on how to structure a real‑world task (e.g., data extraction, summarization, or interactive QA)?\n\nJust let me know which direction you’d like to take (or throw a specific question at me), and we can roll out the code together!']
['2\u202f+\u202f2 equals **4**.']


---
## 4. Caching & `rollout_id`

DSPy **caches all LM calls by default** (to disk via LiteLLM). This saves money and speeds up iteration.

| Parameter | Effect |
|---|---|
| Same input + same `rollout_id` | Returns **cached** result (no new API call) |
| Same input + new `rollout_id` | Forces a **fresh** API call, results are cached under new ID |
| `temperature=0` + new `rollout_id` | Still no effect — temperature must be >0 for varied outputs |

> **When to use:** Use `rollout_id` + `temperature > 0` to explore diverse outputs without burning the cache.

In [6]:
# rollout_id=1 → new independent call, result will be cached under id=1
r1 = lm("Where did the first lunar landing happen?", rollout_id=1, temperature=1.0)
print("Call 1:", r1)

# Same input, same rollout_id=1 → returns CACHED result
r2 = lm("Where did the first lunar landing happen?", rollout_id=1, temperature=1.0)
print("Call 2 (cached):", r2)

# Same input, new rollout_id=2 → FRESH API call
r3 = lm("Where did the first lunar landing happen?", rollout_id=2, temperature=1.0)
print("Call 3 (fresh):", r3)

Call 1: ['The first human‑made lunar landing took place on **July\u202f20,\u202f1969, in the Sea of Tranquility (Mare\u202fTranquillitatis) on the Moon**. Astronauts Neil\u202fA. Armstrong and Buzz\u202fA. Aldrin touched down in the Apollo\xa011 lunar module “Eagle” at roughly **latitude\u202f0.67408°\u202fN, longitude\u202f23.47297°\u202fE**.']
Call 2 (cached): ['The first human‑made lunar landing took place on **July\u202f20,\u202f1969, in the Sea of Tranquility (Mare\u202fTranquillitatis) on the Moon**. Astronauts Neil\u202fA. Armstrong and Buzz\u202fA. Aldrin touched down in the Apollo\xa011 lunar module “Eagle” at roughly **latitude\u202f0.67408°\u202fN, longitude\u202f23.47297°\u202fE**.']
Call 3 (fresh): ['The first manned lunar landing (Apollo\u202f11, July\u202f20\u202f1969) touched down in the **Sea of Tranquility** (Mare Tranquillitatis) on the Moon, near 0.674\u202f°\u202fN latitude and 23.473\u202f°\u202fE longitude.']


In [7]:
# lm.history stores ALL calls with full metadata
print(f"Total calls in history: {len(lm.history)}")
print("Keys in last call:", list(lm.history[-1].keys()))

Total calls in history: 5
Keys in last call: ['prompt', 'messages', 'kwargs', 'response', 'outputs', 'usage', 'cost', 'timestamp', 'uuid', 'model', 'response_model', 'model_type']


### Checking Cache Hits in History

In [8]:
def check_cache_hit(history_item: dict) -> bool:
    """Check if a history entry was served from cache."""
    if 'cache_hit' in history_item:
        return history_item['cache_hit']
    response = history_item.get('response')
    if response and hasattr(response, 'cache_hit'):
        return getattr(response, 'cache_hit', False)
    return False

for i, item in enumerate(lm.history):
    status = "CACHED  ✓" if check_cache_hit(item) else "NEW API call"
    model_used = item.get('model', 'unknown')
    print(f"  Call {i+1}: [{status}] | model={model_used}")

  Call 1: [CACHED  ✓] | model=groq/openai/gpt-oss-20b
  Call 2: [CACHED  ✓] | model=groq/openai/gpt-oss-20b
  Call 3: [CACHED  ✓] | model=groq/openai/gpt-oss-20b
  Call 4: [CACHED  ✓] | model=groq/openai/gpt-oss-20b
  Call 5: [CACHED  ✓] | model=groq/openai/gpt-oss-20b


---
## 5. Signatures — Inline

A **Signature** tells DSPy *what* task to do (input fields → output fields), without specifying *how* to prompt the LM.  
DSPy compiles signatures into optimized prompts automatically.

### Syntax
```python
"input_field -> output_field"
"field1, field2 -> out1, out2"
"field: type -> output: type"
```

### Common Patterns
| Task | Signature |
|---|---|
| Q&A | `"question -> answer"` |
| Summarization | `"document -> summary"` |
| Sentiment | `"sentence -> sentiment: bool"` |
| RAG Q&A | `"context: list[str], question -> answer"` |
| MCQ | `"question, choices: list[str] -> reasoning, selection: int"` |

In [9]:
# --- Example 1: Basic Q&A ---
pred_qa = Predict('question -> answer')
result = pred_qa(question="Where did the first lunar landing happen?")
print("Answer:", result.answer)

# The result is a dspy.Prediction object — access fields by name
print("Type:", type(result))

Answer: The first lunar landing occurred on July 20, 1969, when Apollo 11's lunar module, Eagle, touched down on the Moon's Sea of Tranquility.
Type: <class 'dspy.primitives.prediction.Prediction'>


In [10]:
# --- Example 2: Typed output — sentiment as bool ---
classify = dspy.Predict('sentence -> sentiment: bool')
sentence = "it's a charming and often affecting journey."
print("Sentiment (positive=True):", classify(sentence=sentence).sentiment)

# --- Example 3: Summarization ---
summarize = dspy.ChainOfThought('document -> summary')
doc = """The 21-year-old Lee made seven appearances for West Ham and scored once.
He had loan spells at Blackpool and Colchester United, scoring twice for the latter.
He has now signed with Barnsley; contract length undisclosed."""
response = summarize(document=doc)
print("\nSummary:", response.summary)
print("Reasoning:", response.reasoning)  # ChainOfThought adds a reasoning field

Sentiment (positive=True): True

Summary: Lee, 21, has played 7 times for West Ham, scoring once, and had loan spells at Blackpool and Colchester United (2 goals). He has now joined Barnsley, with contract details undisclosed.
Reasoning: The document contains a brief biographical and career update about a football player named Lee. Key information that can be extracted includes:

1. **Personal Detail**: Lee is 21 years old.
2. **West Ham Tenure**: He has made 7 appearances for West Ham and scored 1 goal.
3. **Loan Spells**:
   - Blackpool (no statistical detail provided).
   - Colchester United (scored 2 goals).
4. **Current Status**: He has signed with Barnsley; the length of the contract is not disclosed.

These details are sufficient to create a concise summary that captures Lee’s age, recent playing record, loan experience, and latest club move.


In [11]:
# --- Example 4: Multiple outputs ---
pred_multi = dspy.Predict("Question, Context -> Answer, Confidence")
res = pred_multi(
    Question="What is the capital of France?",
    Context="Basic geography knowledge"
)
print("Answer:", res.Answer)
print("Confidence:", res.Confidence)

Answer: Paris
Confidence: High


In [12]:
# --- Example 5: Signature with instructions (inline) ---
toxicity = dspy.Predict(
    dspy.Signature(
        "comment -> toxic: bool",
        instructions="Mark as 'toxic' if the comment contains insults, harassment, or sarcastic derogatory remarks.",
    )
)
print("Is toxic:", toxicity(comment="you are beautiful!").toxic)
print("Is toxic:", toxicity(comment="You are absolutely useless, go away!").toxic)

Is toxic: False
Is toxic: True


In [13]:
# --- Example 6: Override LM config per-call with config= ---
pred = dspy.Predict("Question, Context -> Answer, Confidence")

res = pred(
    Question="What is the capital of France?",
    Context="Basic geography knowledge",
    config={"rollout_id": 5, "temperature": 1.0}  # per-call override
)
print("Answer:", res.Answer)
print("Confidence:", res.Confidence)

Answer: Paris
Confidence: High


---
## 6. Signatures — Class-based (Advanced)

Use **class-based signatures** when you need:
1. A **docstring** to clarify the task for the LM  
2. **`desc=`** on fields to add hints or constraints  
3. **Typed outputs** (Literal, pydantic, bool, float, dict, list, etc.)

```python
class MySignature(dspy.Signature):
    """Docstring clarifies the task."""
    input_field: type = dspy.InputField(desc="hint about this input")
    output_field: type = dspy.OutputField(desc="constraint on output")
```

In [14]:
# --- Class-based: Confident Q&A ---
class ScoredAnswer(dspy.Signature):
    """Answer the question based on context and rate confidence from 0.0 to 1.0."""

    Question: str = dspy.InputField()
    Context:  str = dspy.InputField(desc="Background information to ground the answer")
    Answer:   str = dspy.OutputField()
    Confidence: float = dspy.OutputField(desc="A float between 0.0 (unsure) and 1.0 (certain)")

pred_scored = dspy.Predict(ScoredAnswer)
res = pred_scored(Question="What is the capital of France?", Context="Geography 101")
print(f"Answer:     {res.Answer}")
print(f"Confidence: {res.Confidence}")

Answer:     Paris
Confidence: 1.0


In [15]:
from typing import Literal

# --- Class-based: Multi-class emotion classification ---
class Emotion(dspy.Signature):
    """Classify the emotion expressed in the sentence."""

    sentence: str = dspy.InputField()
    sentiment: Literal['sadness', 'joy', 'love', 'anger', 'fear', 'surprise'] = dspy.OutputField()

classify_emotion = dspy.Predict(Emotion)

tests = [
    "i started feeling vulnerable when the spotlight blinded me",
    "I just got promoted! This is the best day ever!",
    "How dare you do this to me!",
]
for s in tests:
    result = classify_emotion(sentence=s)
    print(f"  '{s[:50]}...' → {result.sentiment}")

  'i started feeling vulnerable when the spotlight bl...' → fear
  'I just got promoted! This is the best day ever!...' → joy
  'How dare you do this to me!...' → anger


---
## 7. History Inspection

DSPy provides two ways to look at past LM calls:

| Method | Use case |
|---|---|
| `lm.history` | Raw list of dicts — full metadata (tokens, cost, messages, timestamps) |
| `lm.inspect_history(n=3)` | Human-readable summary of last `n` calls — great for debugging |

# How many calls have been made so far?
print(f"Total history entries: {len(lm.history)}")

# All available metadata keys for each call:
print("Metadata keys:", list(lm.history[-1].keys()))

# Cost and token usage for the last call:
last = lm.history[-1]
print(f"Model:  {last.get('model')}")
print(f"Cost:   ${last.get('cost', 0):.6f}")
print(f"Usage:  {last.get('usage')}")

In [16]:
# Human-readable view of the last 3 calls (prompt + response)
lm.inspect_history(n=3)





[2026-02-18T13:29:11.815720]

System message:

Your input fields are:
1. `sentence` (str):
Your output fields are:
1. `sentiment` (Literal['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']):
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## sentence ## ]]
{sentence}

[[ ## sentiment ## ]]
{sentiment}        # note: the value you produce must exactly match (no extra characters) one of: sadness; joy; love; anger; fear; surprise

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Classify the emotion expressed in the sentence.


User message:

[[ ## sentence ## ]]
i started feeling vulnerable when the spotlight blinded me

Respond with the corresponding output fields, starting with the field `[[ ## sentiment ## ]]` (must be formatted as a valid Python Literal['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']), and then ending with the marker for `[[ ## completed ## ]]`.


Response:

[[ ## sen

In [17]:
# Cache audit — check every call
for i, item in enumerate(lm.history):
    status = "CACHED  ✓" if check_cache_hit(item) else "new call"
    print(f"  Call {i+1:02d}: [{status}]")

  Call 01: [CACHED  ✓]
  Call 02: [CACHED  ✓]
  Call 03: [CACHED  ✓]
  Call 04: [CACHED  ✓]
  Call 05: [CACHED  ✓]
  Call 06: [CACHED  ✓]
  Call 07: [CACHED  ✓]
  Call 08: [CACHED  ✓]
  Call 09: [CACHED  ✓]
  Call 10: [CACHED  ✓]
  Call 11: [CACHED  ✓]
  Call 12: [CACHED  ✓]
  Call 13: [CACHED  ✓]
  Call 14: [CACHED  ✓]
  Call 15: [CACHED  ✓]
  Call 16: [CACHED  ✓]


---
## 8. Responses API

By default, DSPy uses **LiteLLM's Chat Completions API**.  
For certain OpenAI reasoning models (e.g., `gpt-5`, `o3`), you can unlock extra features via the **Responses API** by setting `model_type="responses"`.

> **Note:** Not all providers support the Responses API — check [LiteLLM docs](https://docs.litellm.ai/docs/response_api).

In [18]:
# Standard way (Chat Completions API — default)
lm_chat = dspy.LM(
    model=model,
    api_base=api_base,
    api_key=groq_api_key,
    # model_type defaults to "chat"
)

# Responses API variant (for OpenAI reasoning models like gpt-5 / o3)
# lm_responses = dspy.LM(
#     model="openai/gpt-5-mini",
#     model_type="responses",
#     temperature=1.0,
#     max_tokens=16000,
# )

# The only difference is model_type= — everything else is the same
print("Chat LM ready:", lm_chat.model)

Chat LM ready: groq/openai/gpt-oss-20b


---
## 9. Cheatsheet

```python
import dspy

# ── Setup ──────────────────────────────────────────────────────────
lm = dspy.LM("openai/gpt-4o-mini", api_key="...")
dspy.configure(lm=lm)

# ── Direct call ────────────────────────────────────────────────────
lm("Say hello!")                         # list of strings
lm("Hi", rollout_id=1, temperature=1.0)  # fresh call, cached under id=1

# ── Signatures ─────────────────────────────────────────────────────
# Inline
pred = dspy.Predict("question -> answer")
pred(question="What is AI?").answer

# Inline with types
pred = dspy.Predict("sentence -> sentiment: bool")

# Inline with instructions
pred = dspy.Predict(dspy.Signature("comment -> toxic: bool",
    instructions="Mark toxic if it contains harassment."))

# Class-based
class Q(dspy.Signature):
    """Answer questions accurately."""
    question: str = dspy.InputField()
    answer:   str = dspy.OutputField()
    score: float  = dspy.OutputField(desc="0.0–1.0 confidence")

pred = dspy.Predict(Q)
res = pred(question="What is the moon?")

# ── Modules ────────────────────────────────────────────────────────
dspy.Predict(sig)              # basic LM call
dspy.ChainOfThought(sig)       # adds reasoning field before answer
dspy.ProgramOfThought(sig)     # outputs code; execution = answer
dspy.ReAct(sig, tools=[...])   # agent with tool use
dspy.MultiChainComparison(sig) # compares N outputs → best answer

# Per-call config override
pred(question="...", config={"temperature": 1.0, "rollout_id": 3})

# ── History ────────────────────────────────────────────────────────
len(lm.history)          # number of calls
lm.history[-1].keys()   # metadata: prompt, model, usage, cost, ...
lm.inspect_history(n=3) # human-readable last n calls
```